# 3 · Pull: negotiate a contract, then fetch the data

The consumer and the provider negotiate a contract over the chosen dataset step by step (DSP contract negotiation), then the consumer opens a pull transfer and fetches the data through the provider's dataplane. The provider calls its backend with its own credentials; the consumer never sees them.

In [ ]:
%run ./02_catalog.ipynb

In [ ]:
steps = []   # every DSP step of this notebook, for the timeline at the end


def rpc(base, path, payload, title=None):
    """POST one DSP step to an agent, show the call and keep it for the timeline."""
    response = requests.post(base + path, headers=json_header, data=json.dumps(payload))
    body = show(response, title)
    steps.append({
        "agent": helpers.AGENTS.get(base, base),
        "step": path.split("/dsp/current/", 1)[-1],
        "state": str(helpers.state_of(body) or "").rsplit(":", 1)[-1],
        "ms": int(response.elapsed.total_seconds() * 1000),
    })
    return body


def timeline(title):
    table(steps, [("Agent", "agent"), ("Step", "step"), ("State", "state"), ("ms", "ms")], title=title)


# The offer goes back exactly as the catalog advertises it.
offer = {
    "@id": policy_id,
    "@type": "Offer",
    "target": target_id,
    "permission": policy.get("permission") or [],
    "obligation": policy.get("obligation") or [],
    "prohibition": policy.get("prohibition") or [],
}

def show_data(body, title):
    """A list of records as a table (its first columns), anything else as JSON."""
    if isinstance(body, list) and body and all(isinstance(r, dict) for r in body):
        keys = [k for k, v in body[0].items() if not isinstance(v, (dict, list))][:7]
        table(body[:20], [(k, k) for k in keys], title=f"{title} · {len(body)} records" + (" (first 20)" if len(body) > 20 else ""))
    else:
        print(json.dumps(body, indent=2, ensure_ascii=False)[:4000])

# Contract negotiation

```
REQUESTED → OFFERED → REQUESTED → OFFERED → ACCEPTED → AGREED → VERIFIED → FINALIZED
```

Each step is a call to the agent that sends the message: it signs it and delivers it to the other agent over DSP. In a real negotiation the terms can change at each offer; here they stay as published.

In [ ]:
payload = {
    "associatedAgentPeer": provider_did,
    "providerAddress": provider_dsp,
    "callbackAddress": consumer_dsp,
    "offer": offer,
}
body = rpc(consumer, "/dsp/current/negotiations/rpc/setup-request-init", payload, "1 · Contract request")
pids = {"consumerPid": body["response"]["consumerPid"], "providerPid": body["response"]["providerPid"]}

In [ ]:
rpc(provider, "/dsp/current/negotiations/rpc/setup-offer", {**pids, "offer": offer}, "2 · Offer")
rpc(consumer, "/dsp/current/negotiations/rpc/setup-request", {**pids, "offer": offer}, "3 · Request on the offer")
rpc(provider, "/dsp/current/negotiations/rpc/setup-offer", {**pids, "offer": offer}, "4 · Final offer");

In [ ]:
rpc(consumer, "/dsp/current/negotiations/rpc/setup-acceptance", pids, "5 · Acceptance")
rpc(provider, "/dsp/current/negotiations/rpc/setup-agreement", pids, "6 · Agreement")
rpc(consumer, "/dsp/current/negotiations/rpc/setup-verification", pids, "7 · Verification")
body = rpc(provider, "/dsp/current/negotiations/rpc/setup-finalization", pids, "8 · Finalization")
agreement_id = body["negotiationAgentModel"]["agreement"]["id"]
timeline("The negotiation, step by step");

# Pull transfer

## The consumer requests a transfer on the agreement, the provider starts it

The start message carries the data address: a dataplane URL on the provider, valid for this transfer only.

In [ ]:
if "HTTP_PULL" not in formats(dataset):
    raise RuntimeError(f"{dataset.get('title')} offers no HTTP_PULL distribution: {formats(dataset)}")

payload = {
    "associatedAgentPeer": provider_did,
    "providerAddress": provider_dsp,
    "callbackAddress": consumer_dsp,
    "agreementId": agreement_id,
    "format": "HTTP_PULL",
}
body = rpc(consumer, "/dsp/current/transfers/rpc/setup-request", payload, "Transfer request")
transfer = {"consumerPid": body["response"]["consumerPid"], "providerPid": body["response"]["providerPid"]}

from urllib.parse import urlsplit, urlunsplit

body = rpc(provider, "/dsp/current/transfers/rpc/setup-start", transfer, "Transfer start")
# The agent writes its container address; from here the provider is at its own URL.
parts, here = urlsplit(helpers.find_dataplane(body)), urlsplit(provider)
dataplane = urlunsplit((here.scheme, here.netloc, parts.path, parts.query, parts.fragment))
helpers.AGENTS[dataplane] = f"{provider_name}'s dataplane"
print(f"Dataplane  {dataplane}")

## The consumer pulls the data

In [ ]:
response = requests.get(dataplane, timeout=90)
data = show(response, "GET through the dataplane", open_response=False)
show_data(data, "What came through")

## Suspend, restart, complete

While suspended the dataplane refuses; after completion it is closed for good. A new pull needs a new transfer, on the same agreement.

In [ ]:
rpc(consumer, "/dsp/current/transfers/rpc/setup-suspension", {**transfer, "code": "SUSPEND", "reason": ["Suspended from the notebook"]}, "Suspension")
show(requests.get(dataplane, timeout=90), "GET through the dataplane, while suspended", check=False)
rpc(consumer, "/dsp/current/transfers/rpc/setup-start", transfer, "Restart")
rpc(provider, "/dsp/current/transfers/rpc/setup-completion", transfer, "Completion")
timeline("Everything this notebook asked of the agents");